In [1]:
import torch
import torch.nn as nn

In [2]:
# ----- Plain RNN -----
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h = self.rnn(x)    # out: (batch, seq_len, hidden_size)
        out = self.fc(out[:, -1, :])  # take last timestep only
        return out

In [3]:
# ----- LSTM — one word changed -----
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)  # ← LSTM
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, (h, c) = self.lstm(x)   # LSTM returns h AND cell state c
        out = self.fc(out[:, -1, :]) # still just use last timestep
        return out

In [4]:
# Toy example: classify a sequence of numbers as "going up" or "going down"
model  = LSTMModel(input_size=1, hidden_size=32, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Input: (batch=4, seq_len=5, features=1)
x = torch.randn(4, 5, 1)   # 4 sequences, each 5 steps long, 1 feature per step
y = torch.randint(0, 2, (4,))  # 4 labels: 0 or 1

output = model(x)           # shape: (4, 2)
loss   = criterion(output, y)
loss.backward()
optimizer.step()

print("Output shape:", output.shape)  # (4, 2) — one prediction per sequence

Output shape: torch.Size([4, 2])
